# Backward improvement

**Table of contents**<a id='toc0_'></a>    
- 1. [Imports](#toc1_)    
- 2. [1D](#toc2_)    
- 3. [2D](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 1. <a id='toc1_'></a>[Imports](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch

import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({"axes.grid":True,"grid.color": "black","grid.alpha":"0.25","grid.linestyle": "--"})
plt.rcParams.update({'font.size':14})
plt.rcParams.update({'font.family':'serif'})

from EconDLSolvers import choose_gpu
from NonConvexDurablesModel import NonConvexDurablesModelClass

/work/RaphaëlPaulXavierHuleux#0575/anaconda/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
device, _ = choose_gpu()

GPU 0: 79.18GB free [NVIDIA H100 80GB HBM3]
Best GPU: 0


## 2. <a id='toc2_'></a>[1D](#toc0_)

In [ ]:
model_simult = NonConvexDurablesModelClass(device=device,
                                           load='../output/NonConvexDurablesModel_DL_1D_sigmoid.pt')

In [6]:
print(model_simult.sim.R)

tensor(-16.8485, device='cuda:0')


In [ ]:
learning_rate_policy_t = torch.ones((model_simult.par.T),device=device) * 1e-4
learning_rate_policy_decay_t = torch.ones((model_simult.par.T),device=device)
Nepochs_policy_t = torch.ones((model_simult.par.T),device=device,dtype=torch.int64) * 500

learning_rate_value_t = torch.ones((model_simult.par.T),device=device) * 1e-4
learning_rate_value_decay_t = torch.ones((model_simult.par.T),device=device)
Nepochs_value_t = torch.ones((model_simult.par.T),device=device,dtype=torch.int64) * 500

K_time = 120.0 

train = {}
train['backward'] = True
train['use_simult_in_backward'] = True
train['NN_init_std'] = 0.001
train['Nneurons_value_t'] = np.array([50,50])
train['Nneurons_policy_t'] = np.array([50,50])
train['N'] = 1_000_000
train['N_target_batches'] = 100
train['batch_size'] = 100_000
train['K_time'] = K_time

model_backward = NonConvexDurablesModelClass(device=device,algoname='DeepVPDDCBackward',train=train, par={'D':1,'full':True})
model_backward.solve(model_simult=model_simult,do_print=True)
model_backward.simulate_Rs()

started solving: 2026-02-06 11:28:56
t =  19
 training policy network 
  epoch =     0: 1.10941e+00 
  epoch =   100: 1.10941e+00 
  epoch =   200: 1.10941e+00 
  epoch =   300: 1.10941e+00 
  epoch =   400: 1.10941e+00 
  epoch =   500: 1.10941e+00 
  epoch =   600: 1.10941e+00 
  epoch =   700: 1.10941e+00 
  epoch =   800: 1.10941e+00 
  epoch =   900: 1.10941e+00 
  epoch =   999: 1.10941e+00 
t =  18
 training value network 
  epoch =     0: 3.1e-06
  epoch =   162: 1.7e-06 time limit reached [181.0 secs]
 training policy network 
  epoch =     0: 2.11431e+00 
  epoch =   100: 2.11382e+00 
  epoch =   200: 2.11382e+00 
  epoch =   300: 2.11382e+00 
  epoch =   335:     2.1 time limit reached [180.5 secs]
t =  17
 training value network 
  epoch =     0: 6.6e-06
  epoch =    33: 3.4e-06 time limit reached [183.0 secs]
 training policy network 
  epoch =     0: 3.09733e+00 
  epoch =   100: 3.09732e+00 
  epoch =   200: 3.09732e+00 
  epoch =   300: 3.09732e+00 
  epoch =   333:    

In [ ]:
model_backward.more_simulation_outcomes()
model_backward.euler_errors_DL()

In [ ]:
save = True 
if save:
    vars = ['reward', 'actions', 'DC', 'outcomes', 'taste_shocks', 'shocks', 'states_pd', 'states']
    for var in vars:
        setattr(model_backward.train, var, None)
    
    model_backward.save('../output/NonConvexDurablesModel_DL_1D_backward.pt')

## 3. <a id='toc3_'></a>[2D](#toc0_)

In [ ]:
import gc

# deep learning
gc.collect()

# 3. Clear PyTorch CUDA cache
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
device, _ = choose_gpu()

GPU 0: 42.75GB free [NVIDIA L40]
Best GPU: 0


In [ ]:
model_simult_2D = NonConvexDurablesModelClass(device=device, load='../output/NonConvexDurablesModel_DL_2D_sigmoid.pt')

In [ ]:
print(model_simult_2D.sim.R)

tensor(-18.1597, device='cuda:0')


In [ ]:
model_backward_2D = NonConvexDurablesModelClass(device=device,algoname='DeepVPDDCBackward',train=train, par={'D':2,'full':True})
model_backward_2D.solve(model_simult=model_simult_2D,do_print=True)
model_backward_2D.simulate_Rs()

started solving: 2026-02-03 13:48:55
t =  19
 training policy network 
  epoch =     0: 1.25628e+00 
  epoch =   100: 1.25628e+00 
  epoch =   200: 1.25628e+00 
  epoch =   300: 1.25628e+00 
  epoch =   400: 1.25628e+00 
  epoch =   500: 1.25628e+00 
  epoch =   600: 1.25628e+00 
  epoch =   700: 1.25628e+00 
  epoch =   800: 1.25628e+00 
  epoch =   900: 1.25628e+00 
  epoch =   999: 1.25628e+00 
t =  18
 training value network 
  epoch =     0: 3.5e-06
  epoch =   110: 1.8e-06 time limit reached [180.6 secs]
 training policy network 
  epoch =     0: 2.32981e+00 
  epoch =   100: 2.32979e+00 
  epoch =   112:     2.3 time limit reached [180.7 secs]
t =  17
 training value network 
  epoch =     0: 6.1e-06
  epoch =    12: 2.5e-06 time limit reached [193.5 secs]
 training policy network 
  epoch =     0: 3.38336e+00 
  epoch =   100: 3.38334e+00 
  epoch =   112:     3.4 time limit reached [180.7 secs]
t =  16
 training value network 
  epoch =     0: 6.4e-06
  epoch =    12: 3.5e-06 

array([-18.15278625, -18.16964531, -18.14605522, -18.15695   ,
       -18.16408539])

In [ ]:
model_backward_2D.more_simulation_outcomes()
model_backward_2D.euler_errors_DL()

In [ ]:
save = True 
if save:
    # empty train namesapce to save memory
    vars = ['reward', 'actions', 'DC', 'outcomes', 'taste_shocks', 'shocks', 'states_pd', 'states']
    for var in vars:
        setattr(model_backward_2D.train, var, None)

    model_backward_2D.save('../output/NonConvexDurablesModel_DL_2D_backward.pt')